# Validation A.1 — soda-lime hemisphere workflow

This is an end-to-end teaching example, **not a literature validation**. By default it reads the committed normal-incidence TE spectrum and obtains the equilibrium temperature, IV curve, maximum power, fill factor, efficiency, and figures. An optional cell can rebuild that spectrum with S4.

**Learning goals:** separate the optics and thermal stages; inspect the main PV outputs; understand why a fast smoke test is not a convergence study.

## 1. Prepare the temporary Colab runtime

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shlex
import subprocess
import sys

from IPython.display import Image, Markdown, display

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("Open this notebook in Google Colab before running setup.")

PROJECT_DIR = Path("/content/radcoolpv-py")

def run_command(args: list[str], cwd: Path | None = None, capture: bool = False):
    print("$", shlex.join(args))
    return subprocess.run(
        args, cwd=cwd, check=True, text=True,
        capture_output=capture,
    )

if not PROJECT_DIR.exists():
    run_command([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/gsilvaoelker/radcoolpv-py.git",
        str(PROJECT_DIR),
    ])

run_command([
    sys.executable, "-m", "pip", "install", "--quiet", "--editable", ".",
], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)

print("Repository:", PROJECT_DIR)

## 2. Optional: rebuild the optical spectrum

Leave `RUN_LIVE_OPTICS = False` for the fast classroom path. Setting it to `True` compiles the supported S4 revision and runs 2000 wavelength points with 10 Fourier modes. That is a smoke test, not a converged calculation. The close-packed pitch is an assumption because the paper value is not recorded in the repository.

In [ ]:
import importlib
import importlib.util

S4_DIR = Path("/content/S4")
S4_COMMIT = "9569f5e555b967a4324eb1ea593d0f9f40761a61"

def install_s4() -> None:
    """Build the supported phoebe-p/S4 revision in this Colab runtime."""
    if importlib.util.find_spec("S4") is not None:
        print("S4 is already importable.")
        return
    run_command(["apt-get", "-qq", "update"])
    run_command([
        "apt-get", "-qq", "install", "-y", "build-essential", "git",
        "libboost-all-dev", "libfftw3-dev", "liblapack-dev",
        "libopenblas-dev", "libsuitesparse-dev",
    ])
    if not S4_DIR.exists():
        run_command(["git", "clone", "https://github.com/phoebe-p/S4.git", str(S4_DIR)])
    run_command(["git", "checkout", S4_COMMIT], cwd=S4_DIR)
    run_command(["make", "-j2", "S4_pyext"], cwd=S4_DIR)
    importlib.invalidate_caches()
    import S4
    print("S4:", S4.__file__)

In [ ]:
RUN_LIVE_OPTICS = False

if RUN_LIVE_OPTICS:
    install_s4()
    run_command([
        "radcoolpv", "run",
        "validations/validation A.1/optics_hemisph_sodalime.yaml",
    ], cwd=PROJECT_DIR)
else:
    print("Using the committed optical spectrum; S4 was not run.")

## 3. Obtain PV parameters and figures

In [ ]:
from radcoolpv import config as config_module
from radcoolpv import pipeline

config_path = PROJECT_DIR / "validations/validation A.1/pv_hemisph_sodalime.yaml"
cfg = config_module.load(str(config_path))
cfg.run.plots = True
context = pipeline.run(cfg)

manifest = json.loads((Path(context.results_dir) / "run.json").read_text())
thermal = context.thermal
display(Markdown(
    "| Result | Value |\n|---|---:|\n"
    f"| Equilibrium temperature | {thermal.equil_temp:.2f} K |\n"
    f"| Short-circuit current density | {thermal.isc:.2f} A m$^{{-2}}$ |\n"
    f"| Open-circuit voltage | {thermal.voc_equil:.4f} V |\n"
    f"| Maximum power | {thermal.mpp_equil:.2f} W m$^{{-2}}$ |\n"
    f"| Fill factor | {thermal.ff_equil:.3f} |\n"
    f"| Efficiency | {thermal.efficiency_equil:.4f} |"
))

for figure in sorted((Path(context.results_dir) / "figures").glob("*.png")):
    display(Image(filename=str(figure)))

## 4. Interpret before changing parameters

The reference result is approximately 317.09 K and 230.61 W/m². Agreement with those numbers checks execution of the stored-spectrum workflow only. The thermal model treats a normal-incidence TE spectrum as angle-independent, and the 10-mode S4 option has no convergence evidence.

**Exercise:** copy the PV YAML, change `thermal.convection_coefficient` from 12.0 to 8.0 W/m²/K, and predict the direction of the temperature and power changes before running it.

**Reference geometry context:** G. Silva-Oelker and J. Jaramillo-Fernandez (2022), [doi:10.1364/OE.466335](https://doi.org/10.1364/OE.466335).